In [ ]:
# =====================================================================
# Code created by RootrootQ — 2026
# This code is intended to be used as provided, and no known issues are
# expected during normal use. Please use it according to the requirements
# and limitations of the TRELLIS.2 project and the runtime environment.
# YouTube: https://www.youtube.com/@RootrootQ
# =====================================================================
#
# =====================================================================
# TRELLIS.2 — COLAB A100 40GB
# FINAL ONE-CELL INSTALL + MEMORY-SAFE RUN
# =====================================================================
#
# OBJECTIVE:
#   - Single Colab cell
#   - NO kernel restart
#   - NO Flash-Attention source build
#   - PREBUILT Flash-Attention
#   - A100 40GB
#   - Torch 2.6.0 + CUDA 12.4
#   - low_vram = False
#   - offload = False
#   - Official CUDA extensions
#   - DINOv3 Transformers 5.x compatibility
#   - OpenEXR compatibility
#   - RAM/VRAM cleanup
#   - Single heavy-operation queue for Gradio
#   - Official app.py remains unchanged
#   - app_memory_safe.py is generated automatically
#
# WHEN RUN AGAIN:
#   - Existing environment is preserved
#   - Existing Torch installation is preserved
#   - Flash-Attention is not rebuilt
#   - CUDA extensions are not rebuilt
#   - Model cache is preserved
#   - Only missing components are installed
#
# =====================================================================

import os
import sys
import time
import subprocess
import shutil
import re
import json
import gc
import ctypes
from pathlib import Path


# =====================================================================
# CONFIG
# =====================================================================

BASE = Path("/content")

REPO = BASE / "TRELLIS.2"

ENV_DIR = BASE / "trellis2_env"

EXT_DIR = BASE / "trellis2_extensions"

WHEEL_DIR = BASE / "trellis2_wheels"

PYTHON = ENV_DIR / "bin" / "python"

PIP = ENV_DIR / "bin" / "pip"

MODEL_ID = "microsoft/TRELLIS.2-4B"

TORCH_VERSION = "2.6.0"

TORCHVISION_VERSION = "0.21.0"

TORCH_INDEX = (
    "https://download.pytorch.org/whl/cu124"
)

OPENCV_VERSION = "4.10.0.84"

FLASH_VERSION = "2.7.3"

FLASH_WHEEL_NAME = (
    "flash_attn-2.7.3+cu12torch2.6"
    "cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
)

FLASH_WHEEL_URL = (
    "https://github.com/Dao-AILab/"
    "flash-attention/releases/download/v2.7.3/"
    + FLASH_WHEEL_NAME
)

FLASH_WHEEL = (
    WHEEL_DIR / FLASH_WHEEL_NAME
)

OFFICIAL_APP = REPO / "app.py"

SAFE_APP = REPO / "app_memory_safe.py"

EXTRACTOR = (
    REPO
    / "trellis2"
    / "modules"
    / "image_feature_extractor.py"
)

T0 = time.time()


# =====================================================================
# ENVIRONMENT
# =====================================================================

os.environ["CUDA_MODULE_LOADING"] = "LAZY"

os.environ["PYTHONUNBUFFERED"] = "1"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

os.environ["GRADIO_SHARE"] = "true"

os.environ["MAX_JOBS"] = "4"

os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"

ENV = os.environ.copy()


# =====================================================================
# LOG
# =====================================================================

def ts():
    return f"[{time.time() - T0:7.1f}s]"


def section(title):
    print()
    print("=" * 76)
    print(f"{ts()} {title}")
    print("=" * 76)
    sys.stdout.flush()


def log(msg):
    print(
        f"{ts()} {msg}",
        flush=True
    )


# =====================================================================
# COMMAND RUNNER
# =====================================================================

def run(
    cmd,
    cwd=None,
    env=None,
    check=True,
):

    print(
        f"{ts()} $ {cmd}",
        flush=True
    )

    p = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []

    for line in p.stdout:

        line = line.rstrip()

        print(
            "    " + line,
            flush=True
        )

        tail.append(line)

        if len(tail) > 120:
            tail.pop(0)

    p.wait()

    if check and p.returncode != 0:

        raise RuntimeError(
            "\n"
            "============================================================\n"
            "COMMAND FAILED\n"
            "============================================================\n"
            f"{cmd}\n\n"
            + "\n".join(tail[-80:])
        )

    return p.returncode


# =====================================================================
# PYTHON HELPERS
# =====================================================================

def env_python(code):

    return subprocess.run(
        [
            str(PYTHON),
            "-c",
            code,
        ],
        capture_output=True,
        text=True,
        env=ENV,
    )


def env_import(module):

    p = subprocess.run(
        [
            str(PYTHON),
            "-c",
            f"import {module}",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=ENV,
    )

    return p.returncode == 0


def env_pip(*packages):

    if not packages:
        return

    cmd = (
        f'"{PIP}" install '
        + " ".join(
            f'"{x}"'
            for x in packages
        )
    )

    run(
        cmd,
        env=ENV,
    )


# =====================================================================
# [0/10] GPU
# =====================================================================

section(
    "[0/10] GPU / Runtime check"
)

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

if gpu.returncode != 0:

    raise RuntimeError(
        "NVIDIA GPU not found."
    )

GPU_INFO = gpu.stdout.strip()

log(
    f"GPU: {GPU_INFO}"
)

log(
    "Host Python: "
    + sys.version.split()[0]
)

if "A100" not in GPU_INFO:

    raise RuntimeError(
        "This installer is designed for an A100 40GB GPU.\n"
        f"Detected GPU: {GPU_INFO}"
    )


# =====================================================================
# [1/10] SYSTEM
# =====================================================================

section(
    "[1/10] System tools"
)

run(
    "apt-get -qq update",
    check=False,
)

run(
    "apt-get -qq install -y "
    "git "
    "git-lfs "
    "wget "
    "curl "
    "build-essential "
    "ninja-build "
    "libjpeg-dev "
    "pkg-config "
    "zlib1g-dev "
    "libgl1 "
    "libglib2.0-0",
)

run(
    "git lfs install --force"
)


# =====================================================================
# CUDA
# =====================================================================

section(
    "[CUDA] CUDA Toolkit"
)

if not shutil.which("nvcc"):

    raise RuntimeError(
        "nvcc not found."
    )

run(
    "nvcc --version"
)

CUDA_HOME = Path(
    "/usr/local/cuda"
)

if CUDA_HOME.exists():

    ENV["CUDA_HOME"] = (
        str(CUDA_HOME)
    )

    ENV["PATH"] = (
        str(CUDA_HOME / "bin")
        + ":"
        + ENV.get("PATH", "")
    )

log(
    f"CUDA_HOME = {CUDA_HOME}"
)


# =====================================================================
# [2/10] REPOSITORY
# =====================================================================

section(
    "[2/10] TRELLIS.2 repository"
)

os.chdir(BASE)

if (
    REPO.exists()
    and
    not (REPO / ".git").exists()
):

    log(
        "Removing corrupted repository..."
    )

    shutil.rmtree(
        REPO,
        ignore_errors=True
    )

if not REPO.exists():

    run(
        "git clone "
        "--depth 1 "
        "--recursive "
        "-b main "
        "https://github.com/microsoft/TRELLIS.2.git "
        f'"{REPO}"'
    )

else:

    log(
        "TRELLIS.2 repository already exists."
    )

os.chdir(REPO)

run(
    "git rev-parse --short HEAD",
    cwd=REPO,
)


# =====================================================================
# [3/10] ISOLATED ENVIRONMENT
# =====================================================================

section(
    "[3/10] Isolated Python environment"
)

virtualenv_check = subprocess.run(
    [
        sys.executable,
        "-m",
        "virtualenv",
        "--version",
    ],
    capture_output=True,
    text=True,
)

if virtualenv_check.returncode != 0:

    log(
        "Installing virtualenv..."
    )

    run(
        f'"{sys.executable}" -m pip install '
        "virtualenv "
        "--quiet "
        "--disable-pip-version-check"
    )

if (
    ENV_DIR.exists()
    and
    not PYTHON.exists()
):

    log(
        "Removing corrupted environment..."
    )

    shutil.rmtree(
        ENV_DIR,
        ignore_errors=True
    )

if not PYTHON.exists():

    run(
        f'"{sys.executable}" -m virtualenv '
        "--no-download "
        f'"{ENV_DIR}"'
    )

else:

    log(
        "Isolated environment already exists."
    )

run(
    f'"{PYTHON}" -m pip install '
    "--upgrade "
    "pip setuptools wheel "
    "--quiet "
    "--disable-pip-version-check",
    env=ENV,
)

version = subprocess.run(
    [
        str(PYTHON),
        "--version",
    ],
    capture_output=True,
    text=True,
    env=ENV,
)

log(
    version.stdout.strip()
)


# =====================================================================
# TORCH
# =====================================================================

section(
    "[Torch] PyTorch 2.6.0 + CUDA 12.4"
)

torch_test = env_python(
    """
import torch

print(torch.__version__)
print(torch.version.cuda)
"""
)

torch_ready = (
    torch_test.returncode == 0
    and
    "2.6.0+cu124"
    in torch_test.stdout
)

if not torch_ready:

    log(
        "Installing PyTorch 2.6.0 + CUDA 12.4..."
    )

    env_pip(
        f"torch=={TORCH_VERSION}",
        f"torchvision=={TORCHVISION_VERSION}",
        "--index-url",
        TORCH_INDEX,
        "--no-cache-dir",
    )

torch_test = env_python(
    """
import torch

print(
    "TORCH =",
    torch.__version__
)

print(
    "CUDA =",
    torch.version.cuda
)

print(
    "CUDA_OK =",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU =",
        torch.cuda.get_device_name(0)
    )

    print(
        "CC =",
        torch.cuda.get_device_capability(0)
    )

    print(
        "ABI =",
        torch._C._GLIBCXX_USE_CXX11_ABI
    )
"""
)

if torch_test.returncode != 0:

    print(
        torch_test.stderr,
        flush=True
    )

    raise RuntimeError(
        "PyTorch import failed."
    )

print(
    torch_test.stdout,
    flush=True
)

if (
    "2.6.0+cu124"
    not in torch_test.stdout
):

    raise RuntimeError(
        "PyTorch 2.6.0 + CUDA 12.4 "
        "could not be verified."
    )

if (
    "CUDA_OK = True"
    not in torch_test.stdout
):

    raise RuntimeError(
        "PyTorch cannot detect the CUDA GPU."
    )

ABI_TEST = env_python(
    """
import torch
print(
    "TRUE"
    if torch._C._GLIBCXX_USE_CXX11_ABI
    else
    "FALSE"
)
"""
)

ABI = ABI_TEST.stdout.strip()

log(
    f"PyTorch ABI = {ABI}"
)

if ABI != "FALSE":

    raise RuntimeError(
        "Bu prebuilt Flash-Attention wheel "
        "is built for CXX11 ABI FALSE."
    )


# =====================================================================
# [4/10] FLASH ATTENTION
# =====================================================================

section(
    "[4/10] Flash-Attention — PREBUILT ONLY"
)

if env_import(
    "flash_attn"
):

    flash_info = env_python(
        """
import flash_attn

print(
    getattr(
        flash_attn,
        "__version__",
        "unknown"
    )
)

print(
    flash_attn.__file__
)
"""
    )

    print(
        "Flash-Attention already installed:",
        flash_info.stdout,
        flush=True
    )

else:

    WHEEL_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    if not FLASH_WHEEL.exists():

        log(
            "Downloading prebuilt Flash-Attention wheel..."
        )

        run(
            f'curl -L --fail --retry 3 '
            f'-o "{FLASH_WHEEL}" '
            f'"{FLASH_WHEEL_URL}"'
        )

    else:

        log(
            "Flash wheel cache already exists."
        )

    wheel_size = (
        FLASH_WHEEL.stat().st_size
        / 1024**2
    )

    log(
        f"Flash wheel: "
        f"{wheel_size:.1f} MB"
    )

    if wheel_size < 100:

        raise RuntimeError(
            "Flash wheel eksik/bozuk."
        )

    # ================================================================
    # CRITICAL:
    # NO SOURCE BUILD
    # ================================================================

    run(
        f'"{PIP}" install '
        f'"{FLASH_WHEEL}" '
        "--no-deps",
        env=ENV,
    )

FLASH_TEST = env_python(
    """
import flash_attn

print(
    "VERSION =",
    getattr(
        flash_attn,
        "__version__",
        "unknown"
    )
)

print(
    "PATH =",
    flash_attn.__file__
)
"""
)

if FLASH_TEST.returncode != 0:

    print(
        FLASH_TEST.stderr,
        flush=True
    )

    raise RuntimeError(
        "Flash-Attention import failed."
    )

print(
    FLASH_TEST.stdout,
    flush=True
)

FLASH_CUDA = env_python(
    """
import torch
from flash_attn import flash_attn_func

q = torch.randn(
    1, 128, 8, 64,
    device="cuda",
    dtype=torch.float16,
)

k = torch.randn_like(q)
v = torch.randn_like(q)

y = flash_attn_func(
    q, k, v
)

torch.cuda.synchronize()

print("FLASH_ATTN_CUDA_OK")
print("OUTPUT =", tuple(y.shape))
"""
)

if FLASH_CUDA.returncode != 0:

    print(
        FLASH_CUDA.stdout
    )

    print(
        FLASH_CUDA.stderr
    )

    raise RuntimeError(
        "Flash-Attention CUDA test failed."
    )

print(
    FLASH_CUDA.stdout,
    flush=True
)


# =====================================================================
# [5/10] PYTHON DEPENDENCIES
# =====================================================================

section(
    "[5/10] Python dependencies"
)

env_pip(
    "imageio",
    "imageio-ffmpeg",
    "tqdm",
    "easydict",
    f"opencv-python-headless=={OPENCV_VERSION}",
    "ninja",
    "trimesh",
    "transformers",
    "gradio==6.0.1",
    "tensorboard",
    "pandas",
    "lpips",
    "zstandard",
    "kornia",
    "timm",
    "psutil",
)


# =====================================================================
# utils3d
# =====================================================================

if not env_import(
    "utils3d"
):

    env_pip(
        "git+https://github.com/"
        "EasternJournalist/utils3d.git@"
        "9a4eb15e4021b67b12c460c7057d642626897ec8"
    )

else:

    log(
        "utils3d: OK"
    )


# =====================================================================
# OPENCV / EXR
# =====================================================================

section(
    "[OpenCV] OpenEXR test"
)

EXR_TEST = env_python(
    r"""
import os

os.environ[
    "OPENCV_IO_ENABLE_OPENEXR"
] = "1"

import cv2

path = (
    "/content/TRELLIS.2/"
    "assets/hdri/forest.exr"
)

print(
    "OPENCV =",
    cv2.__version__
)

print(
    "EXR_READER =",
    cv2.haveImageReader(path)
)

img = cv2.imread(
    path,
    cv2.IMREAD_UNCHANGED
)

if img is None:
    raise RuntimeError(
        "Could not read forest.exr."
    )

print("EXR_OK")
print("SHAPE =", img.shape)
print("DTYPE =", img.dtype)
"""
)

if EXR_TEST.returncode != 0:

    print(
        EXR_TEST.stdout
    )

    print(
        EXR_TEST.stderr
    )

    raise RuntimeError(
        "OpenEXR test failed."
    )

print(
    EXR_TEST.stdout,
    flush=True
)


# =====================================================================
# [6/10] CUDA EXTENSIONS
# =====================================================================

section(
    "[6/10] TRELLIS.2 CUDA extensions"
)

EXT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ENV["MAX_JOBS"] = "4"

ENV["TORCH_CUDA_ARCH_LIST"] = "8.0"


# ---------------------------------------------------------------------
# nvdiffrast
# ---------------------------------------------------------------------

if env_import(
    "nvdiffrast"
):

    log(
        "nvdiffrast: OK"
    )

else:

    path = (
        EXT_DIR / "nvdiffrast"
    )

    if not path.exists():

        run(
            "git clone "
            "-b v0.4.0 "
            "https://github.com/NVlabs/nvdiffrast.git "
            f'"{path}"'
        )

    run(
        f'"{PIP}" install '
        f'"{path}" '
        "--no-build-isolation",
        env=ENV,
    )


# ---------------------------------------------------------------------
# nvdiffrec
# ---------------------------------------------------------------------

if env_import(
    "nvdiffrec_render"
):

    log(
        "nvdiffrec: OK"
    )

else:

    path = (
        EXT_DIR / "nvdiffrec"
    )

    if not path.exists():

        run(
            "git clone "
            "-b renderutils "
            "https://github.com/"
            "JeffreyXiang/nvdiffrec.git "
            f'"{path}"'
        )

    run(
        f'"{PIP}" install '
        f'"{path}" '
        "--no-build-isolation",
        env=ENV,
    )


# ---------------------------------------------------------------------
# CuMesh
# ---------------------------------------------------------------------

if env_import(
    "cumesh"
):

    log(
        "CuMesh: OK"
    )

else:

    path = (
        EXT_DIR / "CuMesh"
    )

    if not path.exists():

        run(
            "git clone "
            "--recursive "
            "https://github.com/"
            "JeffreyXiang/CuMesh.git "
            f'"{path}"'
        )

    run(
        f'"{PIP}" install '
        f'"{path}" '
        "--no-build-isolation",
        env=ENV,
    )


# ---------------------------------------------------------------------
# FlexGEMM
# ---------------------------------------------------------------------

if env_import(
    "flex_gemm"
):

    log(
        "FlexGEMM: OK"
    )

else:

    path = (
        EXT_DIR / "FlexGEMM"
    )

    if not path.exists():

        run(
            "git clone "
            "--recursive "
            "https://github.com/"
            "JeffreyXiang/FlexGEMM.git "
            f'"{path}"'
        )

    run(
        f'"{PIP}" install '
        f'"{path}" '
        "--no-build-isolation",
        env=ENV,
    )


# ---------------------------------------------------------------------
# o-voxel
# ---------------------------------------------------------------------

if env_import(
    "o_voxel"
):

    log(
        "o-voxel: OK"
    )

else:

    source = (
        REPO / "o-voxel"
    )

    run(
        f'"{PIP}" install '
        f'"{source}" '
        "--no-build-isolation",
        env=ENV,
    )


# =====================================================================
# [7/10] HDRI
# =====================================================================

section(
    "[7/10] HDRI assets"
)

HDRI_DIR = (
    REPO
    / "assets"
    / "hdri"
)

HDRI_DIR.mkdir(
    parents=True,
    exist_ok=True
)

run(
    "git lfs install --force",
    cwd=REPO,
    check=False
)

exrs = list(
    HDRI_DIR.glob("*.exr")
)

if not exrs:

    log(
        "HDRI eksik → Git LFS pull"
    )

    run(
        'git lfs pull --include="assets/hdri/*"',
        cwd=REPO,
        check=False
    )

exrs = list(
    HDRI_DIR.glob("*.exr")
)

if not exrs:

    raise RuntimeError(
        "HDRI files not found."
    )

for f in exrs:

    log(
        f"{f.name}: "
        f"{f.stat().st_size / 1024**2:.2f} MB"
    )


# =====================================================================
# [8/10] DINOv3
# =====================================================================

section(
    "[8/10] DINOv3 compatibility"
)

if not EXTRACTOR.exists():

    raise RuntimeError(
        "image_feature_extractor.py not found."
    )

source = EXTRACTOR.read_text(
    encoding="utf-8"
)

# ---------------------------------------------------------------------
# DINOv3:
#
# Transformers 5.x:
#     self.model.model.layer
#
# Older version:
#     self.model.layer
#
# Upstream issue #147 confirms this.
# ---------------------------------------------------------------------

method_pattern = re.compile(
    r"(?ms)^    def extract_features\(.*?"
    r"(?=^    @torch\.no_grad\(\))"
)

method_match = method_pattern.search(
    source
)

if not method_match:

    raise RuntimeError(
        "DINOv3 extract_features() "
        "function not found."
    )

NEW_DINO_METHOD = r'''    def extract_features(self, image: torch.Tensor) -> torch.Tensor:
        image = image.to(
            self.model.embeddings.patch_embeddings.weight.dtype
        )

        hidden_states = self.model.embeddings(
            image,
            bool_masked_pos=None
        )

        position_embeddings = self.model.rope_embeddings(
            image
        )

        encoder = getattr(
            self.model,
            "model",
            self.model
        )

        layers = getattr(
            encoder,
            "layer",
            None
        )

        if layers is None:
            raise AttributeError(
                "DINOv3 encoder layer list not found."
            )

        for i, layer_module in enumerate(
            layers
        ):
            hidden_states = layer_module(
                hidden_states,
                position_embeddings=position_embeddings,
            )

        return F.layer_norm(
            hidden_states,
            hidden_states.shape[-1:]
        )

'''

source = (
    source[:method_match.start()]
    + NEW_DINO_METHOD
    + source[method_match.end():]
)

EXTRACTOR.write_text(
    source,
    encoding="utf-8"
)

# Syntax
compile_test = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "py_compile",
        str(EXTRACTOR),
    ],
    capture_output=True,
    text=True,
    env=ENV,
)

if compile_test.returncode != 0:

    print(
        compile_test.stderr
    )

    raise RuntimeError(
        "DINOv3 syntax error."
    )

log(
    "DINO extractor syntax OK."
)

dino_import = subprocess.run(
    [
        str(PYTHON),
        "-c",
        """
from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor
print("DINOv3_IMPORT_OK")
""",
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
    env=ENV,
)

if dino_import.returncode != 0:

    print(
        dino_import.stderr
    )

    raise RuntimeError(
        "DINOv3 import failed."
    )

print(
    dino_import.stdout,
    end="",
    flush=True
)


# =====================================================================
# [9/10] HUGGING FACE
# =====================================================================

section(
    "[9/10] Hugging Face model"
)

try:

    from google.colab import userdata

    HF_TOKEN = userdata.get(
        "HF_TOKEN"
    )

except Exception:

    HF_TOKEN = None

if not HF_TOKEN:

    raise RuntimeError(
        "In Colab Secrets, "
        "HF_TOKEN was not found."
    )

ENV["HF_HOME"] = (
    "/root/.cache/huggingface"
)

ENV["HF_TOKEN"] = HF_TOKEN

ENV[
    "HUGGING_FACE_HUB_TOKEN"
] = HF_TOKEN

HF_CODE = r"""
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="microsoft/TRELLIS.2-4B",
    repo_type="model",
    local_files_only=False,
)

print(
    "MODEL_PATH=" + str(path)
)
"""

hf = subprocess.run(
    [
        str(PYTHON),
        "-c",
        HF_CODE,
    ],
    cwd=str(REPO),
    env=ENV,
    capture_output=True,
    text=True,
)

print(
    hf.stdout,
    end="",
    flush=True
)

if hf.returncode != 0:

    print(
        hf.stderr
    )

    raise RuntimeError(
        "Hugging Face model cache failed."
    )

MODEL_PATH = None

for line in hf.stdout.splitlines():

    if line.startswith(
        "MODEL_PATH="
    ):

        MODEL_PATH = Path(
            line.split(
                "=",
                1
            )[1].strip()
        )

        break

if MODEL_PATH is None:

    raise RuntimeError(
        "Model path not found."
    )

log(
    f"Model cache: {MODEL_PATH}"
)

PIPELINE_JSON = (
    MODEL_PATH
    / "pipeline.json"
)

if PIPELINE_JSON.exists():

    cfg = json.loads(
        PIPELINE_JSON.read_text(
            encoding="utf-8"
        )
    )

    cfg["low_vram"] = False

    PIPELINE_JSON.write_text(
        json.dumps(
            cfg,
            indent=2
        ),
        encoding="utf-8"
    )

    log(
        "low_vram = FALSE"
    )


# =====================================================================
# [10/10] FINAL SYSTEM TEST
# =====================================================================

section(
    "[10/10] Final system test"
)

MODULES = [
    "torch",
    "torchvision",
    "transformers",
    "gradio",
    "cv2",
    "trimesh",
    "utils3d",
    "flash_attn",
    "nvdiffrast",
    "cumesh",
    "flex_gemm",
    "o_voxel",
    "psutil",
]

for module in MODULES:

    if not env_import(
        module
    ):

        raise RuntimeError(
            f"Import failed: {module}"
        )

    log(
        f"{module}: OK"
    )

FINAL = env_python(
    r"""
import torch
import cv2
import flash_attn

print(
    "TORCH =",
    torch.__version__
)

print(
    "CUDA =",
    torch.version.cuda
)

print(
    "FLASH_ATTN =",
    getattr(
        flash_attn,
        "__version__",
        "unknown"
    )
)

print(
    "OPENCV =",
    cv2.__version__
)

print(
    "GPU =",
    torch.cuda.get_device_name(0)
)

print(
    "CC =",
    torch.cuda.get_device_capability(0)
)

free_vram, total_vram = (
    torch.cuda.mem_get_info()
)

print(
    "FREE_VRAM_GB =",
    round(
        free_vram / 1024**3,
        2
    )
)

print(
    "TOTAL_VRAM_GB =",
    round(
        total_vram / 1024**3,
        2
    )
)
"""
)

if FINAL.returncode != 0:

    print(
        FINAL.stderr
    )

    raise RuntimeError(
        "Final GPU test failed."
    )

print(
    FINAL.stdout,
    flush=True
)

TRELLIS_TEST = env_python(
    """
import trellis2
print("TRELLIS2_IMPORT_OK")
"""
)

if TRELLIS_TEST.returncode != 0:

    print(
        TRELLIS_TEST.stderr
    )

    raise RuntimeError(
        "TRELLIS2 import failed."
    )

print(
    TRELLIS_TEST.stdout,
    end="",
    flush=True
)


# =====================================================================
# MEMORY SAFE APP
# =====================================================================

section(
    "[APP] Creating memory-safe app"
)

if not OFFICIAL_APP.exists():

    raise RuntimeError(
        "Official app.py not found."
    )

OFFICIAL = OFFICIAL_APP.read_text(
    encoding="utf-8"
)

# ---------------------------------------------------------------------
# Kesin upstream signature.
#
# This is the actual signature in Microsoft TRELLIS.2 app.py:
#
# image_to_3d(...,
#     req: gr.Request,
#     progress=gr.Progress(track_tqdm=True),
# )
#
# extract_glb(...,
#     req: gr.Request,
#     progress=gr.Progress(track_tqdm=True),
# )
#
# ---------------------------------------------------------------------

IMAGE_DEF_RE = re.compile(
    r"(?ms)^def image_to_3d\(.*?"
    r"(?=^def extract_glb\()"
)

image_match = IMAGE_DEF_RE.search(
    OFFICIAL
)

if not image_match:

    raise RuntimeError(
        "Official image_to_3d() not found."
    )

# ---------------------------------------------------------------------
# Preserve the original function by renaming it.
# ---------------------------------------------------------------------

OFFICIAL_SAFE = (
    OFFICIAL[:image_match.start()]
    + "def _original_image_to_3d(\n"
    + OFFICIAL[
        image_match.start()
        + len("def image_to_3d("):
        image_match.end()
    ]
)

# The mechanism above is revalidated below using a marker
# to ensure reliable function-body separation.
#
# Safer approach:
# use the official source with only the function name changed.
# ---------------------------------------------------------------------

OFFICIAL_SAFE = OFFICIAL.replace(
    "def image_to_3d(",
    "def _original_image_to_3d(",
    1
)

# extract_glb'yi de koru
OFFICIAL_SAFE = OFFICIAL_SAFE.replace(
    "def extract_glb(",
    "def _original_extract_glb(",
    1
)

# ---------------------------------------------------------------------
# Wrapper memory layer
# ---------------------------------------------------------------------

MEMORY_LAYER = r'''

# ============================================================
# TRELLIS2 MEMORY SAFETY LAYER
# ============================================================

import gc
import ctypes
import threading
import psutil

_MEMORY_LOCK = threading.RLock()


def _trim_ram():

    try:

        libc = ctypes.CDLL(
            "libc.so.6"
        )

        malloc_trim = getattr(
            libc,
            "malloc_trim",
            None
        )

        if malloc_trim:

            malloc_trim(0)

    except Exception:

        pass


def _report_memory(tag):

    try:

        process = psutil.Process()

        process_ram = (
            process.memory_info().rss
            / 1024**3
        )

        ram = psutil.virtual_memory()

        system_used = (
            ram.used
            / 1024**3
        )

        system_total = (
            ram.total
            / 1024**3
        )

        print(
            "[MEMORY] "
            f"{tag} | "
            f"Process={process_ram:.2f} GB | "
            f"System={system_used:.2f}/"
            f"{system_total:.2f} GB",
            flush=True
        )

    except Exception:

        pass


def _cleanup_cuda(tag):

    try:

        gc.collect()

    except Exception:

        pass

    try:

        if torch.cuda.is_available():

            torch.cuda.synchronize()

            torch.cuda.empty_cache()

            try:

                torch.cuda.ipc_collect()

            except Exception:

                pass

    except Exception:

        pass

    _trim_ram()

    try:

        if torch.cuda.is_available():

            free_vram, total_vram = (
                torch.cuda.mem_get_info()
            )

            print(
                "[MEMORY] "
                f"{tag} | "
                f"Free VRAM="
                f"{free_vram / 1024**3:.2f}/"
                f"{total_vram / 1024**3:.2f} GB",
                flush=True
            )

    except Exception:

        pass


# ============================================================
# IMAGE -> 3D
#
# IMPORTANT:
# req and progress are explicit.
# ============================================================

def image_to_3d(
    image: Image.Image,
    seed: int,
    resolution: str,

    ss_guidance_strength: float,
    ss_guidance_rescale: float,
    ss_sampling_steps: int,
    ss_rescale_t: float,

    shape_slat_guidance_strength: float,
    shape_slat_guidance_rescale: float,
    shape_slat_sampling_steps: int,
    shape_slat_rescale_t: float,

    tex_slat_guidance_strength: float,
    tex_slat_guidance_rescale: float,
    tex_slat_sampling_steps: int,
    tex_slat_rescale_t: float,

    req: gr.Request,

    progress=gr.Progress(
        track_tqdm=True
    ),
):

    with _MEMORY_LOCK:

        print(
            "[MEMORY] "
            "Generation lock acquired.",
            flush=True
        )

        _report_memory(
            "generation start"
        )

        try:

            # ------------------------------------------------
            # DO NOT cleanup during generation.
            #
            # Emptying CUDA cache every sampling step would
            # destroy performance.
            #
            # Cleanup happens ONLY after request completes.
            # ------------------------------------------------

            result = _original_image_to_3d(
                image,
                seed,
                resolution,

                ss_guidance_strength,
                ss_guidance_rescale,
                ss_sampling_steps,
                ss_rescale_t,

                shape_slat_guidance_strength,
                shape_slat_guidance_rescale,
                shape_slat_sampling_steps,
                shape_slat_rescale_t,

                tex_slat_guidance_strength,
                tex_slat_guidance_rescale,
                tex_slat_sampling_steps,
                tex_slat_rescale_t,

                req,
                progress,
            )

            return result

        finally:

            _cleanup_cuda(
                "generation complete"
            )

            _report_memory(
                "generation cleanup complete"
            )


# ============================================================
# GLB
# ============================================================

def extract_glb(
    state: dict,
    decimation_target: int,
    texture_size: int,
    req: gr.Request,
    progress=gr.Progress(
        track_tqdm=True
    ),
):

    with _MEMORY_LOCK:

        print(
            "[MEMORY] "
            "GLB lock acquired.",
            flush=True
        )

        _report_memory(
            "GLB start"
        )

        try:

            result = _original_extract_glb(
                state,
                decimation_target,
                texture_size,
                req,
                progress,
            )

            return result

        finally:

            _cleanup_cuda(
                "GLB complete"
            )

            _report_memory(
                "GLB cleanup complete"
            )

'''

# ---------------------------------------------------------------------
# Insert layer BEFORE with gr.Blocks
# ---------------------------------------------------------------------

BLOCK_MARKER = (
    "with gr.Blocks(delete_cache=(600, 600)) as demo:"
)

if BLOCK_MARKER not in OFFICIAL_SAFE:

    raise RuntimeError(
        "Gradio Blocks marker not found."
    )

SAFE_SOURCE = (
    OFFICIAL_SAFE.replace(
        BLOCK_MARKER,
        MEMORY_LAYER
        + "\n\n"
        + BLOCK_MARKER,
        1
    )
)

# ---------------------------------------------------------------------
# Concurrency control.
#
# Do not run two heavy CUDA generations at the same time.
# This is the safest behavior because this model uses a single GPU.
# ---------------------------------------------------------------------

LAUNCH = (
    "demo.launch(css=css, head=head)"
)

QUEUE_LAUNCH = (
    "demo.queue("
    "max_size=1, "
    "default_concurrency_limit=1"
    ").launch(css=css, head=head)"
)

if "demo.queue(" not in SAFE_SOURCE:

    if LAUNCH in SAFE_SOURCE:

        SAFE_SOURCE = SAFE_SOURCE.replace(
            LAUNCH,
            QUEUE_LAUNCH,
            1
        )

SAFE_APP.write_text(
    SAFE_SOURCE,
    encoding="utf-8"
)

log(
    f"Memory-safe app: {SAFE_APP}"
)


# =====================================================================
# APP COMPILE
# =====================================================================

APP_COMPILE = subprocess.run(
    [
        str(PYTHON),
        "-m",
        "py_compile",
        str(SAFE_APP),
    ],
    capture_output=True,
    text=True,
    env=ENV,
)

if APP_COMPILE.returncode != 0:

    print(
        APP_COMPILE.stdout
    )

    print(
        APP_COMPILE.stderr
    )

    raise RuntimeError(
        "app_memory_safe.py syntax error."
    )

log(
    "Memory-safe app syntax OK."
)


# =====================================================================
# APP IMPORT CHECK
# =====================================================================

APP_IMPORT = subprocess.run(
    [
        str(PYTHON),
        "-c",
        """
import app_memory_safe
print("APP_IMPORT_OK")
""",
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
    env=ENV,
)

print(
    APP_IMPORT.stdout,
    end="",
    flush=True
)

if APP_IMPORT.returncode != 0:

    print(
        APP_IMPORT.stderr
    )

    raise RuntimeError(
        "Memory-safe app import failed."
    )


# =====================================================================
# FINAL STATUS
# =====================================================================

print()
print("=" * 76)
print(" TRELLIS.2 — READY")
print("=" * 76)

print(
    "GPU             : NVIDIA A100 40GB"
)

print(
    "Python          : isolated 3.12"
)

print(
    "PyTorch         : 2.6.0 + CUDA 12.4"
)

print(
    "Flash-Attn      : 2.7.3 PREBUILT"
)

print(
    "Flash Build     : DISABLED"
)

print(
    "Kernel Restart  : NONE"
)

print(
    "nvdiffrast      : OK"
)

print(
    "nvdiffrec       : OK"
)

print(
    "CuMesh          : OK"
)

print(
    "FlexGEMM        : OK"
)

print(
    "o-voxel         : OK"
)

print(
    "OpenCV          : 4.10.0.84"
)

print(
    "OpenEXR         : OK"
)

print(
    "DINOv3          : FIXED"
)

print(
    "low_vram        : FALSE"
)

print(
    "Offload         : OFF"
)

print(
    "Official app.py : UNTOUCHED"
)

print(
    "Memory-safe app : ENABLED"
)

print(
    "Generation lock : 1"
)

print(
    "GLB lock        : 1"
)

print(
    "RAM trim        : ON"
)

print(
    "CUDA cleanup    : POST REQUEST"
)

print("=" * 76)


# =====================================================================
# START APP
# =====================================================================

APP_ENV = ENV.copy()

APP_ENV[
    "OPENCV_IO_ENABLE_OPENEXR"
] = "1"

APP_ENV[
    "CUDA_MODULE_LOADING"
] = "LAZY"

APP_ENV[
    "PYTHONUNBUFFERED"
] = "1"

APP_ENV[
    "TOKENIZERS_PARALLELISM"
] = "false"

APP_ENV[
    "GRADIO_SHARE"
] = "true"

APP_ENV[
    "PYTORCH_CUDA_ALLOC_CONF"
] = "expandable_segments:True"

APP_ENV[
    "ATTN_BACKEND"
] = "flash_attn"

APP_ENV[
    "TRELLIS2_MEMORY_SAFE"
] = "1"


section(
    "[APP] Starting TRELLIS.2 memory-safe app"
)

run(
    f'"{PYTHON}" -u "app_memory_safe.py"',
    cwd=REPO,
    env=APP_ENV,
    check=True,
)